# JASPER: Interactive Textile Design Analysis

This notebook provides an interactive interface for exploring Japanese x Sri Lankan textile design patterns.

**Research**: Bridging Cultures Through Design DNA  
**Dataset**: 2,000 textile images (1,000 Japanese, 1,000 Sri Lankan)

## Setup

In [ ]:
# Import libraries
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
sys.path.insert(0, '../src')

from feature_extraction import TextileFeatureExtractor, extract_dataset_features
from statistical_analysis import compare_textile_groups, generate_statistical_report
from visualization import *

# Configuration
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print("✓ Setup complete")

## 1. Load Dataset

Configure your dataset path below:

In [ ]:
# CONFIGURE: Set your dataset path
DATASET_PATH = '../data'  # Adjust to your dataset location

# Optional: Use sample for quick testing (set to None for full dataset)
SAMPLE_SIZE = 50  # Use 50 images per class for quick testing
# SAMPLE_SIZE = None  # Uncomment for full dataset

# Load image paths
file_paths = list(glob.glob(os.path.join(DATASET_PATH, '**/*.png'), recursive=True))
labels = [Path(p).parent.name for p in file_paths]

# Create DataFrame
data_df = pd.DataFrame({'filepath': file_paths, 'label': labels})

# Sample if requested
if SAMPLE_SIZE is not None:
    data_df = data_df.groupby('label').sample(n=SAMPLE_SIZE, random_state=42)
    data_df = data_df.reset_index(drop=True)

print(f"Loaded {len(data_df)} images")
print(f"\nLabel distribution:")
print(data_df['label'].value_counts())

## 2. Extract Features

Extract 44 quantitative features from each textile image.

**Note**: This may take 10-30 minutes for the full dataset (2000 images).

In [ ]:
# Extract features (or load from cache)
FEATURES_CACHE = '../results/extracted_features.csv'

if os.path.exists(FEATURES_CACHE):
    print(f"Loading cached features from {FEATURES_CACHE}")
    features_df = pd.read_csv(FEATURES_CACHE)
    
    # Filter to match current sample if needed
    if SAMPLE_SIZE is not None:
        features_df = features_df[features_df['filepath'].isin(data_df['filepath'])]
else:
    print("Extracting features... (this may take a while)")
    features_df = extract_dataset_features(
        data_df['filepath'].tolist(),
        data_df['label'].tolist(),
        output_csv=FEATURES_CACHE,
        verbose=True
    )

print(f"\n✓ Features extracted: {features_df.shape}")
print(f"  {len(features_df)} images × {len(features_df.columns)-2} features")

# Display sample
features_df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Summary statistics by group
feature_cols = [col for col in features_df.columns if col not in ['filepath', 'label']]

print("Summary Statistics by Group:\n")
summary = features_df.groupby('label')[feature_cols].mean()
print(summary.T)

### 3.1 Color Analysis

In [ ]:
# Visualize color palettes
plot_color_palette_comparison(
    features_df, 
    n_samples=10
)

In [ ]:
# RGB comparison
plot_rgb_comparison(features_df)

### 3.2 Key Metrics Distribution

In [ ]:
# Plot distributions for paper's key metrics
key_metrics = ['warm_cool_score', 'texture_complexity', 'texture_contrast', 'symmetry_score']

plot_feature_distributions(
    features_df,
    features=key_metrics
)

## 4. Statistical Analysis

In [ ]:
# Perform comprehensive statistical comparison
comparison_results = compare_textile_groups(
    features_df,
    group1_label='japanese_textiles',
    group2_label='sri_lankan_textiles',
    alpha=0.05,
    apply_correction=True
)

print(f"Analyzed {len(comparison_results)} features")
print(f"Significant after correction: {comparison_results['significant_after_correction'].sum()}")

# Display top 10 by effect size
comparison_results[[
    'feature', 'cohens_d', 'p_value', 'effect_size', 
    'japanese_textiles_mean', 'sri_lankan_textiles_mean'
]].head(10)

In [ ]:
# Generate statistical report
report = generate_statistical_report(comparison_results)
print(report)

### 4.1 Effect Size Visualization

In [ ]:
# Visualize effect sizes
plot_effect_sizes(comparison_results, top_n=20)

In [ ]:
# Volcano plot
plot_pvalue_volcano(comparison_results)

## 5. Key Differentiators

Identify features with **very large effect sizes** (|Cohen's d| >= 1.2) and high significance (p < 0.001):

In [ ]:
# Filter for very large effect sizes
key_features = comparison_results[
    (abs(comparison_results['cohens_d']) >= 1.2) & 
    (comparison_results['p_value'] < 0.001)
].copy()

print(f"Found {len(key_features)} key differentiating features:\n")

for idx, row in key_features.iterrows():
    print(f"{row['feature']}:")
    print(f"  Cohen's d: {row['cohens_d']:.3f} ({row['effect_size']})")
    print(f"  p-value: {row['p_value']:.4e}")
    print(f"  Japanese: {row['japanese_textiles_mean']:.3f}")
    print(f"  Sri Lankan: {row['sri_lankan_textiles_mean']:.3f}")
    print()

## 6. Export Results

In [ ]:
# Save results
os.makedirs('../results', exist_ok=True)

# Save comparison
comparison_results.to_csv('../results/statistical_comparison.csv', index=False)
print("✓ Saved statistical_comparison.csv")

# Save key features
key_features.to_csv('../results/key_differentiators.csv', index=False)
print("✓ Saved key_differentiators.csv")

# Save report
with open('../results/statistical_report.txt', 'w') as f:
    f.write(report)
print("✓ Saved statistical_report.txt")

print("\n✓ All results saved to ../results/")

## 7. Custom Analysis

Use this section for your own exploration:

In [ ]:
# Example: Compare specific features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature 1: Warm/Cool Score
jp_warm = features_df[features_df['label'] == 'japanese_textiles']['warm_cool_score']
sl_warm = features_df[features_df['label'] == 'sri_lankan_textiles']['warm_cool_score']

axes[0].boxplot([jp_warm, sl_warm], labels=['Japanese', 'Sri Lankan'])
axes[0].set_ylabel('Warm/Cool Score')
axes[0].set_title('Color Temperature Comparison')
axes[0].grid(True, alpha=0.3)

# Feature 2: Texture Complexity  
jp_tex = features_df[features_df['label'] == 'japanese_textiles']['texture_complexity']
sl_tex = features_df[features_df['label'] == 'sri_lankan_textiles']['texture_complexity']

axes[1].boxplot([jp_tex, sl_tex], labels=['Japanese', 'Sri Lankan'])
axes[1].set_ylabel('Texture Complexity')
axes[1].set_title('Texture Complexity Comparison')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()